# Bước 2 (tổ hợp T11) + cấu hình cuối F01 trên Kaggle — Phạm Văn Hoàng Anh Tú (2A202602507)

Phần tiếp theo của `kaggle_buoc2.ipynb`, cùng môi trường Kaggle GPU **T4**, cùng code, cùng công thức nền.
1. **T11** = TrivialAugment + sampler cân bằng (hai yếu tố có Δ dương lớn nhất ở Bước 2), seed 0.
2. **F01** = cấu hình có macro-F1 val seed 0 cao nhất trong {T03, T09, T11} (quy tắc chọn ghi sẵn, chỉ dùng val),
   train 3 seed (0, 1, 2). Checkpoint `best.pt` giữ trong Output để Bước 3–4 dùng; test CHƯA chạy ở đây.
Notebook chạy được từ đầu đến cuối không cần bấm gì (dùng **Save Version → Save & Run All**).

- Code lấy từ GitHub (repo công khai), không cần token.
- Ảnh tải từ Zenodo, kiểm tra MD5, giải nén vào đĩa tạm `/tmp/data` (không nằm trong Output).
- Kết quả nhỏ (log, biểu đồ, logit val, `done.json`) được gói vào `/kaggle/working/kaggle_buoc2b.zip`;
  checkpoint `.pt` để lại trong Output, không cho vào zip.

In [ ]:
# Ô K1 — kiểm tra GPU, cài thư viện, in phiên bản
import os, sys, platform, subprocess
LOCAL_TEST = os.environ.get("LAB_LOCAL_TEST") == "1"   # chỉ Claude dùng khi chạy thử trên CPU; trên Kaggle luôn False
if not LOCAL_TEST:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm", "openpyxl"], check=True)
import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert LOCAL_TEST or torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"

In [ ]:
# Ô K2 — lấy code từ GitHub (nhánh có code đã chạy thử)
ROOT = os.environ.get("LAB_ROOT", "/kaggle")
WORK = f"{ROOT}/working"
REPO = f"{WORK}/repo"
BRANCH = "claude/project-thread-mdpq02"
if not LOCAL_TEST and not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/tubepvhat1604-bot/K4-DAY02-PhamVanHoangAnhTu-2A202602507.git", REPO], check=True)
CODE_DIR = f"{REPO}/submissions/2A202602507_pham_van_hoang_anh_tu/code"
for p in (REPO, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
OUT = f"{WORK}/sub"                      # giống cấu trúc thư mục bài nộp: curves/, predictions/, logs/
RUNS_DIR = f"{WORK}/deepweeds_runs"      # checkpoint + log từng run
for d in ("curves", "predictions", "logs"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
os.chdir(OUT)
import train, dataset
from train import Config, run
print("code:", CODE_DIR)

In [ ]:
# Ô K3 — tải ảnh (Zenodo) + CSV fold 0 (GitHub tác giả), kiểm tra MD5, giải nén ra đĩa tạm
import hashlib, zipfile, urllib.request
DATA_DIR   = os.environ.get("LAB_DATA", "/tmp/data")
LABELS_DIR = f"{DATA_DIR}/labels"
ZIP_LOCAL  = f"{DATA_DIR}/images.zip"
ZIP_URL    = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5    = "b7b30f96d466fba86016aa5a26606e0f"
os.makedirs(LABELS_DIR, exist_ok=True)

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not LOCAL_TEST and not os.path.exists(f"{DATA_DIR}/.images_ok"):
    if not os.path.exists(ZIP_LOCAL):
        print("Tải images.zip từ Zenodo...")
        urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
    got = md5sum(ZIP_LOCAL)
    assert got == ZIP_MD5, f"MD5 sai: {got}"
    print("MD5 đúng:", got)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(f"{DATA_DIR}/images")
    open(f"{DATA_DIR}/.images_ok", "w").close()

BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}.csv", f"{LABELS_DIR}/{name}.csv")

IMAGES_DIR = max((r for r, _, fs in os.walk(f"{DATA_DIR}/images")), key=lambda r: len(os.listdir(r)))
print("IMAGES_DIR =", IMAGES_DIR, "|", sum(f.endswith(".jpg") for f in os.listdir(IMAGES_DIR)), "ảnh .jpg")
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
dataset.check_split(train_df, val_df, test_df, IMAGES_DIR,
                    expected_total=None if LOCAL_TEST else dataset.TOTAL_IMAGES, verbose=False)
print("Kiểm tra chia dữ liệu: ĐẠT")

In [ ]:
# Ô K4 — cấu hình chung: GIỐNG HỆT Ô 0.4 + B2.x của lab_day2.ipynb (ConvNeXt-T, công thức nền T00)
BASE = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=RUNS_DIR, curves_dir=f"{OUT}/curves",
            pred_dir=f"{OUT}/predictions", logs_dir=f"{OUT}/logs", num_workers=2)
if LOCAL_TEST:   # chạy thử trên CPU: mạng không pretrained, ảnh nhỏ, 2 bước mỗi epoch
    BASE.update(expected_total=None, pretrained=False, img_size=64, batch_size=16, epochs=1,
                max_steps_per_epoch=2, num_workers=0)
T_BASE = dict(BASE, backbone="convnext_tiny")
print(Config(**T_BASE))

In [ ]:
# Ô K5 — T11 (tổ hợp) rồi F01 = cấu hình tốt nhất trên val trong {T03, T09, T11}, 3 seed
import json, traceback
from pathlib import Path
CANDS = {"T03": ("trivial", dict(aug="trivial")),
         "T09": ("balanced", dict(sampler="balanced")),
         "T11": ("trivial_balanced", dict(aug="trivial", sampler="balanced"))}
# macro-F1 val seed 0 của T03, T09: từ lần chạy trước (kaggle_buoc2.ipynb, logs/recipes_summary.csv)
VAL_SEED0 = {"T03": 0.9741777980950395, "T09": 0.9709312899741346}

desc, change = CANDS["T11"]
VAL_SEED0["T11"] = run(Config(exp_id="T11", seed=0, desc=desc, **dict(T_BASE, **change)))["val_macro_f1"]
print("macro-F1 val seed 0:", {k: round(v, 4) for k, v in VAL_SEED0.items()})

best_id = max(VAL_SEED0, key=VAL_SEED0.get)
F01_DESC, F01_CHANGE = CANDS[best_id]
print(f"F01 = cấu hình của {best_id} ({F01_DESC}): {F01_CHANGE}")
Path(OUT, "logs", "F01_choice.json").write_text(json.dumps(
    {"chosen_from": best_id, "desc": F01_DESC, "change_vs_T00": F01_CHANGE, "val_seed0": VAL_SEED0}, indent=2))

failed = []
for seed in (0, 1, 2):
    try:
        run(Config(exp_id="F01", seed=seed, desc=F01_DESC, **dict(T_BASE, **F01_CHANGE)))
    except Exception:
        traceback.print_exc()
        failed.append(f"F01 seed{seed}")
print("Run lỗi:", failed or "không có")

In [ ]:
# Ô K6 — bảng tóm tắt T11 + F01
df = pd.concat([train.collect_summaries(RUNS_DIR, "T11"), train.collect_summaries(RUNS_DIR, "F01")])
f01 = df.loc[df.exp_id == "F01", "val_macro_f1"]
print(f"F01 ({len(f01)} seed): macro-F1 val = {f01.mean():.4f} ± {f01.std(ddof=1):.4f} (mean ± std)")
cols = ["exp_id", "seed", "desc", "best_epoch", "val_macro_f1", "val_top1", "val_balanced_acc", "val_ece",
        "f1_chinee", "f1_snake", "f1_negative", "train_time_per_epoch_s"]
df[cols].to_csv(f"{OUT}/logs/final_summary.csv", index=False)
print(df[cols].round(4).to_string(index=False))

In [ ]:
# Ô K7 — gói kết quả nhỏ thành kaggle_buoc2b.zip (KHÔNG có checkpoint .pt) để tải về và đưa lên Drive
import zipfile
from pathlib import Path
zp = Path(WORK) / "kaggle_buoc2b.zip"
keep = {"done.json", "config.json", "history.csv", "val_logits.npy", "val_pred.csv"}
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [*Path(RUNS_DIR).glob("T11/seed*/*"), *Path(RUNS_DIR).glob("F01/seed*/*")]:
        if f.name in keep:
            z.write(f, Path("deepweeds_runs") / f.relative_to(RUNS_DIR))
    for sub in ("curves", "logs"):
        for f in Path(OUT, sub).rglob("*"):
            if f.is_file():
                z.write(f, Path("sub") / f.relative_to(OUT))
with zipfile.ZipFile(zp) as z:
    names = z.namelist()
print(zp, f"{zp.stat().st_size / 1e6:.1f} MB,", len(names), "file;",
      sum(n.startswith("deepweeds_runs/") and n.endswith("done.json") for n in names), "run đã xong")